In [1]:
from datetime import datetime
from pymongo import MongoClient

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

import time
import re


# ==========================================
# CONEXION MONGODB
# ==========================================
client = MongoClient("mongodb://bigdata_mongodb1:27017/")
db = client["proyecto_bigdata"]
coleccion = db["alojamientos"]

print("Conexion MongoDB exitosa!")


# ==========================================
# CIUDADES
# ==========================================
ciudades = [
    "Arica", "Iquique", "Calama", "Antofagasta", "Copiapo",
    "La Serena", "Valparaiso", "Vina del Mar", "Santiago",
    "Rancagua", "Talca", "Chillan", "Concepcion", "Temuco",
    "Valdivia", "Puerto Varas", "Puerto Montt", "Coyhaique",
    "Puerto Natales", "Punta Arenas"
]


# ==========================================
# CONFIG DRIVER
# ==========================================
def configurar_driver():
    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-blink-features=AutomationControlled")
    opciones.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )

    return driver


# ==========================================
# EXTRAER PRECIO
# ==========================================
def extraer_precio(texto):
    nums = re.findall(r'\d+', texto.replace(".", ""))
    if nums:
        return int("".join(nums))
    return 0


# ==========================================
# ZONA GEOGRAFICA
# ==========================================
def obtener_zona(ciudad):
    if ciudad in ["Arica", "Iquique", "Calama", "Antofagasta", "Copiapo"]:
        return "Norte"
    elif ciudad in ["La Serena", "Valparaiso", "Vina del Mar", "Santiago", "Rancagua"]:
        return "Centro"
    elif ciudad in ["Talca", "Chillan", "Concepcion", "Temuco", "Valdivia"]:
        return "Sur"
    else:
        return "Austral"


# ==========================================
# SCRAPER HOSTELWORLD
# ==========================================
def scraper_hostelworld(ciudad):

    ciudad_url = ciudad.replace(" ", "-").lower()
    url = f"https://www.hostelworld.com/es/s/{ciudad_url}/"

    print("=" * 60)
    print("Ciudad:", ciudad)
    print("URL:", url)
    print("=" * 60)

    driver = configurar_driver()
    driver.get(url)

    time.sleep(8)

    # Scroll para cargar más resultados
    last_height = driver.execute_script("return document.body.scrollHeight")

    for _ in range(8):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            break

        last_height = new_height

    # ======================================
    # CAPTURAR CARDS
    # ======================================
    cards = driver.find_elements(By.CSS_SELECTOR, "div.property-card")

    if len(cards) == 0:
        cards = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='property-card']")

    print("Cards detectadas:", len(cards))

    guardados = 0

    for i, card in enumerate(cards):

        try:
            texto = card.text.strip()

            if texto == "":
                continue

            lineas = texto.split("\n")

            nombre = lineas[0]
            precio = 0
            puntuacion = 0
            estrellas = 0
            tipo_habitacion = "Hostel"

            for linea in lineas:

                if "$" in linea or "CLP" in linea:
                    precio = extraer_precio(linea)

                if re.search(r'\d+\.\d+', linea):
                    try:
                        puntuacion = float(re.search(r'\d+\.\d+', linea).group())
                    except:
                        pass

                if "Hostel" in linea or "Hotel" in linea or "Bed and Breakfast" in linea:
                    tipo_habitacion = linea

            # Hostelworld normalmente usa nota sobre 10
            if puntuacion >= 9:
                estrellas = 5
            elif puntuacion >= 8:
                estrellas = 4
            elif puntuacion >= 7:
                estrellas = 3
            elif puntuacion > 0:
                estrellas = 2

            registro = {
                "integrante": "juan-salas",
                "grupo": "G5_Turismo_Hoteleria",

                "nombre": nombre,
                "nombre_hotel": nombre,

                "ciudad": ciudad,
                "zona_geografica": obtener_zona(ciudad),

                "precio": precio,
                "precio_clp": precio,

                "estrellas": estrellas,
                "puntuacion": puntuacion,

                "adultos": 2,
                "adultos_habitacion": 2,

                "noches": 3,

                "tipo_habitacion": tipo_habitacion,

                "url_origen": url,
                "plataforma": "hostelworld.com",

                "fecha": datetime.now(),
                "fecha_captura": datetime.now()
            }

            coleccion.insert_one(registro)

            guardados += 1

            print(f"[{i + 1}] {nombre} | ${precio} | Rating: {puntuacion}")

        except Exception as e:
            print("Error en card:", e)
            continue

    print("Guardados:", guardados)

    driver.quit()


# ==========================================
# MAIN
# ==========================================
antes = coleccion.count_documents({"plataforma": "hostelworld.com"})
print("Antes:", antes)

for ciudad in ciudades:
    try:
        scraper_hostelworld(ciudad)
        print("Esperando...")
        time.sleep(5)

    except Exception as e:
        print(f"Error en {ciudad}: {e}")

despues = coleccion.count_documents({"plataforma": "hostelworld.com"})

print("=" * 60)
print("SCRAPING FINALIZADO")
print("=" * 60)
print("Antes:", antes)
print("Ahora:", despues)
print("Nuevos:", despues - antes)
print("=" * 60)

Conexion MongoDB exitosa!
Antes: 0
Ciudad: Arica
URL: https://www.hostelworld.com/es/s/arica/
Error en Arica: Message: unknown error: cannot find Chrome binary
Stacktrace:
#0 0x5af3543334e3 <unknown>
#1 0x5af354062c76 <unknown>
#2 0x5af354089757 <unknown>
#3 0x5af354088029 <unknown>
#4 0x5af3540c6ccc <unknown>
#5 0x5af3540c647f <unknown>
#6 0x5af3540bdde3 <unknown>
#7 0x5af3540932dd <unknown>
#8 0x5af35409434e <unknown>
#9 0x5af3542f33e4 <unknown>
#10 0x5af3542f73d7 <unknown>
#11 0x5af354301b20 <unknown>
#12 0x5af3542f8023 <unknown>
#13 0x5af3542c61aa <unknown>
#14 0x5af35431c6b8 <unknown>
#15 0x5af35431c847 <unknown>
#16 0x5af35432c243 <unknown>
#17 0x731078bb4ac3 <unknown>

Ciudad: Iquique
URL: https://www.hostelworld.com/es/s/iquique/
Error en Iquique: Message: unknown error: cannot find Chrome binary
Stacktrace:
#0 0x5b66aee294e3 <unknown>
#1 0x5b66aeb58c76 <unknown>
#2 0x5b66aeb7f757 <unknown>
#3 0x5b66aeb7e029 <unknown>
#4 0x5b66aebbcccc <unknown>
#5 0x5b66aebbc47f <unknown>
#6 0

KeyboardInterrupt: 

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import shutil

def configurar_driver():
    opciones = Options()

    # Buscar Chrome/Chromium automáticamente
    chrome_path = (
        shutil.which("google-chrome")
        or shutil.which("chromium")
        or shutil.which("chromium-browser")
    )

    if chrome_path is None:
        raise Exception(
            "No se encontró Chrome/Chromium instalado. "
            "Selenium necesita un navegador real para funcionar."
        )

    print("Chrome encontrado en:", chrome_path)

    opciones.binary_location = chrome_path

    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )

    return driver

In [7]:
!conda install -c conda-forge chromium -y

Solving environment: unsuccessful initial attempt using frozen solve. Retrying with flexible solve.
Solving environment: unsuccessful initial attempt using frozen solve. Retrying with flexible solve.

PackagesNotFoundError: The following packages are not available from current channels:

  - chromium

Current channels:

  - https://conda.anaconda.org/conda-forge/linux-64
  - https://conda.anaconda.org/conda-forge/noarch

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.


